In [1]:
import torch 
import numpy as np 
import h5py
import os
from pathlib import Path
import importlib

import src.binaural_attn_lightning as binaural_lightning 
importlib.reload(binaural_lightning)
import yaml
from pytorch_lightning import Trainer


os.environ["HDF5_USE_FILE_LOCKING"] = "FALSE"

config_path = "config/binaural_attn/dev_voice_and_loc_cue_001.yml"
config = yaml.load(open(config_path, 'r'), Loader=yaml.FullLoader)

config['num_workers'] = 0
config['hparas']['batch_size'] = 2
config['audio']['rep_kwargs']['rep_on_gpu'] = True

module = binaural_lightning.BinauralAttentionModule(config)


/om2/user/imgriff/conda_envs/torch_11_cuda_11_pitch/lib/python3.9/site-packages/tqdm/auto.py:22: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


num_classes={'num_words': 998}
Model performing word task
center_crop=True
binaural=True
Binaural cochleagram
using FIR cochleagram


In [2]:
trainer = Trainer(
    precision=32,
    # default_root_dir='test_log_dump/',
    # val_check_interval=1000,
#     limit_train_batches=0.,
    limit_val_batches=0.0,
    num_nodes=1,
    gpus=1,
    # detect_anomaly=True,
    accelerator="gpu",
)

/om2/user/imgriff/conda_envs/torch_11_cuda_11_pitch/lib/python3.9/site-packages/pytorch_lightning/loops/utilities.py:91: PossibleUserWarning: `max_epochs` was not set. Setting it to 1000 epochs. To train without an epoch limit, set `max_epochs=-1`.
  rank_zero_warn(
GPU available: True, used: True
TPU available: False, using: 0 TPU cores
IPU available: False, using: 0 IPUs
HPU available: False, using: 0 HPUs


In [3]:
trainer.fit(module)

LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]

  | Name             | Type                   | Params
------------------------------------------------------------
0 | audio_transforms | AudioCompose           | 0     
1 | model            | AttnSequentialAttacker | 69.6 M
2 | loss_fn          | CrossEntropyLoss       | 0     
3 | train_acc        | Accuracy               | 0     
4 | valid_acc        | Accuracy               | 0     
5 | test_acc         | Accuracy               | 0     
6 | test_confusion   | Accuracy               | 0     
------------------------------------------------------------
69.6 M    Trainable params
0         Non-trainable params
69.6 M    Total params
278.359   Total estimated model params size (MB)
/om2/user/imgriff/conda_envs/torch_11_cuda_11_pitch/lib/python3.9/site-packages/pytorch_lightning/trainer/connectors/data_connector.py:240: PossibleUserWarning: The dataloader, train_dataloader, does not have many workers which may be a bottleneck. Consider increas

len training set = 239478
Epoch 0:   0%|          | 7/119739 [00:05<24:49:40,  1.34it/s, loss=6.89, v_num=3.06e+7]

/om2/user/imgriff/conda_envs/torch_11_cuda_11_pitch/lib/python3.9/site-packages/pytorch_lightning/trainer/trainer.py:726: UserWarning: Detected KeyboardInterrupt, attempting graceful shutdown...
  rank_zero_warn("Detected KeyboardInterrupt, attempting graceful shutdown...")


In [4]:
!nvidia-smi

Fri May 12 12:35:49 2023       
+-----------------------------------------------------------------------------+
| NVIDIA-SMI 460.106.00   Driver Version: 460.106.00   CUDA Version: 11.2     |
|-------------------------------+----------------------+----------------------+
| GPU  Name        Persistence-M| Bus-Id        Disp.A | Volatile Uncorr. ECC |
| Fan  Temp  Perf  Pwr:Usage/Cap|         Memory-Usage | GPU-Util  Compute M. |
|                               |                      |               MIG M. |
|===============================+======================+======================|
|   0  GeForce GTX 108...  On   | 00000000:02:00.0 Off |                  N/A |
| 25%   42C    P8     9W / 250W |  11044MiB / 11178MiB |      0%      Default |
|                               |                      |                  N/A |
+-------------------------------+----------------------+----------------------+
                                                                               
+-------

In [3]:
x = torch.tensor([[0,3,4,],
                   [torch.nan, torch.nan, torch.nan],
                   [torch.nan, torch.nan, torch.nan]])

# x = x.view(2,3,1,1)

print(x.shape)

x[torch.isnan(x).all(dim=1)] = 0 

torch.argwhere(torch.sum(torch.abs(x), dim=1) == 0).squeeze()

torch.Size([3, 3])


tensor([1, 2])

In [4]:
x = torch.tensor([[0,3,4,]]).view(1,1,-1)
torch.cat([x, x], dim=1).shape

torch.Size([1, 2, 3])

In [4]:
!export HDF5_USE_FILE_LOCKING=FALSE


In [3]:
os.environ["HDF5_USE_FILE_LOCKING"] = "FALSE"

data_pth = Path('/om/user/imgriff/datasets/spatial_audio_pipeline/assets/spatial_attention_v1/train/')

eg_file = 'examples_0079838_0081938.hdf5_chunk000'

h5 = h5py.File(data_pth/eg_file, 'r', swmr=True)

In [5]:
h5.keys()

<KeysViewHDF5 ['index', 'label_loc_target_azim', 'label_loc_target_elev', 'label_talker_int', 'label_word_int', 'loc_cue', 'n_sources', 'n_speech_distractors', 'signal', 'snr', 'sr', 'target', 'voice_cue_center_loc', 'voice_cue_target_loc']>

997

In [4]:
bad_files = []
for file in data_pth.glob("*.hdf5_chunk000"):
    try:
        with h5py.File(file, 'r', swmr=True) as f_handle:
             len(f_handle['signal'])
    except Exception as e:
        bad_files.append(file.stem)


In [ ]:
bad_files

['examples_0995720_0997819',
 'examples_0651310_0653410',
 'examples_0783620_0785719']

In [ ]:
import pickle
word_to_class = pickle.load( open('/om2/user/imgriff/datasets/commonvoice_9/en/cv_word_int_label_dict.pkl', 'rb'))
class_to_word = {v:k for k,v in word_to_class.items()}

In [ ]:
h5.keys()


<KeysViewHDF5 ['index', 'label_loc_target_azim', 'label_loc_target_elev', 'label_talker_int', 'label_word_int', 'loc_cue', 'n_sources', 'n_speech_distractors', 'signal', 'snr', 'sr', 'target', 'voice_cue_center_loc', 'voice_cue_target_loc']>

In [ ]:
eg_ix = 2000

word_int = h5['label_word_int'][eg_ix]
scene = h5['signal'][eg_ix]
cue = h5['voice_cue_target_loc'][eg_ix]
target = h5['target'][eg_ix]
snr = h5['snr'][eg_ix]
samp_rate = h5['sr'][eg_ix]

(2, 125000)

In [ ]:
    slice_scene = slice(idx_start, idx_start + len_scene)


NameError: name 'idx_start' is not defined

In [ ]:
from IPython.display import Audio

In [ ]:
Audio(cue.T, rate=samp_rate)


In [ ]:
# stereo audio shape needs to be n_channel x n_samps
Audio(scene.T, rate=samp_rate)

In [ ]:
print(class_to_word[word_int])
Audio(target.T, rate=samp_rate)


nothing


In [ ]:
## Check mono commonvoice for errors 

os.environ["HDF5_USE_FILE_LOCKING"] = "FALSE"

cv_path = Path('/om2/user/imgriff/datasets/commonvoice_9_en/3000ms/stimSR_50000/cv_9_en/subsets/train_core')

eg2_file = 'train_core_50000Hz_rate_60dB_010.hdf5'

h52 = h5py.File(cv_path/eg2_file, 'r', swmr=True)


In [ ]:
h52

<HDF5 file "train_core_50000Hz_rate_60dB_010.hdf5" (mode r)>

In [ ]:
eg_ix = 200

word_int = h52['label_word_int'][eg_ix]
scene = h52['signal'][eg_ix]
# cue = h52['voice_cue_target_loc'][eg_ix]
# target = h52['target'][eg_ix]
# snr = h52['snr'][eg_ix]
samp_rate = h52['sr'][eg_ix]
print(class_to_word[word_int])

Audio(scene, rate=samp_rate)

removed


In [ ]:
config

{'corpus': {'name': 'spatialized_commonvoice_gise_scenes',
  'cue_type': 'voice_and_location',
  'task': 'word',
  'root': '/om/user/imgriff/datasets/spatial_audio_pipeline/assets/spatial_attention_v1/'},
 'audio': {'rep_type': 'cochlea_filt',
  'rep_kwargs': {'sr': 20000,
   'env_sr': 8000,
   'n_channels': 40,
   'low_lim': 40,
   'use_pad': True,
   'binaural': True,
   'rep_on_gpu': False,
   'center_crop': True,
   'out_dur': 2,
   'impulse_len': 0.25,
   'env_extraction_type': 'Half-wave Rectification',
   'downsampling_type': 'TorchTransformsResample',
   'downsampling_kwargs': {'lowpass_filter_width': 64,
    'rolloff': 0.9475937167399596,
    'resampling_method': 'kaiser_window',
    'beta': 14.769656459379492}},
  'compression_type': 'coch_p3',
  'compression_kwargs': {'scale': 1,
   'offset': 1e-07,
   'clip_value': 5,
   'power': 0.3}},
 'val_metric': 'ACC/val_acc',
 'model_name': 'binaural_attn_cue_voice_and_loc',
 'model': {'num_classes': 998,
  'fc_size': 512,
  'global_